# Analyse de l'influence de la température sur le choix de mobilité

Objectif : resoumettre au **même modèle** (provider fixé) les prompts de sélection d'itinéraire issus de `llm_exchanges.jsonl`  
en faisant varier uniquement la **température** LLM (de 0.0 à 2.0),  
puis comparer les choix de mode de transport produits à chaque niveau de température.

> ⚠️ **Depuis le passage au choix modal probabiliste** (2026-07-29), le modèle ne
> choisit plus une option : il attribue une probabilité à **chacune** (somme = 100).
>
> `chosen_index` est ici reconstruit comme l'**argmax** de cette distribution
> (`probability_compat.decision_index`) : les analyses ci-dessous continuent donc de
> mesurer exactement ce qu'elles mesuraient. Mais ce n'est plus ce que fait la
> simulation, qui **tire au sort** dans la distribution.
>
> **Reformulation à envisager** : l'hésitation du modèle, jusqu'ici estimée en
> répétant les appels et en mesurant la dispersion des choix, est désormais lisible
> **directement** dans un seul appel — colonne `entropy` (0 = certitude, 1 = indécision
> totale). Bien moins bruité, et gratuit en requêtes.


In [ ]:
import sys
import json
import time
import threading
import shutil
import yaml
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import pandas as pd
from tqdm.notebook import tqdm

# ── Chemins ────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../..')
sys.path.insert(0, str(PROJECT_ROOT))

EXCHANGES_FILE = PROJECT_ROOT / 'experiments' / 'current' / 'llm_exchanges.jsonl'
SCHEMAS_FILE   = PROJECT_ROOT / 'llm_module' / 'prompts' / 'schemas.json'
PROMPTS_FILE   = Path('prompts.yaml')

SYSTEM_PROMPT_PREFIX = 'Tu es un expert en mobilité urbaine à Toulouse, France.'
CATEGORY             = 'itinary_multi_agent'
SAMPLE_SIZE          = 100

# ── Persona système ─────────────────────────────────────────────────────────
PERSONA_KEY = 'persona_v3'
with open(PROMPTS_FILE) as f:
    _prompts_cfg = yaml.safe_load(f)
PERSONA_SYSTEM_PROMPT = _prompts_cfg['prompts'][PERSONA_KEY]['content']
print(f'Persona chargé : {PERSONA_KEY!r} ({len(PERSONA_SYSTEM_PROMPT)} caractères)')

# ── Configuration de l'expérience ─────────────────────────────────────────
PROVIDER_NAME = 'mistral'   # Provider fixé — seule la température varie
TEMPERATURES  = [0.0, 0.3, 0.5, 0.7, 1.0, 1.2, 1.5]

OUTPUT_CSV = Path('results') / f'temperature_results_{PROVIDER_NAME}_{PERSONA_KEY}.csv'
print(f'Provider     : {PROVIDER_NAME}')
print(f'Températures : {TEMPERATURES}')
print(f'Output CSV   : {OUTPUT_CSV}')

## 1 — Chargement du JSONL et filtrage des entrées éligibles

In [ ]:
import random

def parse_multiline_jsonl(path: Path) -> list:
    content = path.read_text(encoding='utf-8').strip()
    decoder = json.JSONDecoder()
    entries, pos = [], 0
    while pos < len(content):
        stripped = content[pos:].lstrip()
        pos += len(content[pos:]) - len(stripped)
        if not stripped:
            break
        obj, offset = decoder.raw_decode(stripped)
        entries.append(obj)
        pos += offset
    return entries


all_entries = parse_multiline_jsonl(EXCHANGES_FILE)

eligible = [
    e for e in all_entries
    if e.get('messages', [{}])[0].get('content', '').startswith(SYSTEM_PROMPT_PREFIX)
]

if len(eligible) > SAMPLE_SIZE:
    random.seed(42)
    eligible = random.sample(eligible, SAMPLE_SIZE)

print(f'Entrées totales   : {len(all_entries)}')
print(f'Entrées éligibles : {len(eligible)} (max {SAMPLE_SIZE})')

## 2 — Chargement du schéma de réponse et informations du provider

In [ ]:
from llm_module.adapters.base import get_adapter
from llm_module.settings.models import InternalMessage, InternalRequest
from llm_module.tasks.llm_config import settings

with open(SCHEMAS_FILE) as f:
    RESPONSE_SCHEMA = json.load(f)[CATEGORY]

provider_cfg = settings.providers[PROVIDER_NAME]
RPM_LIMIT    = provider_cfg.rpm_limit

print(f'Provider    : {PROVIDER_NAME}')
print(f'Modèle      : {provider_cfg.default_model}')
print(f'RPM limit   : {RPM_LIMIT}')
print(f'Appels total: {len(TEMPERATURES) * len(eligible)} ({len(TEMPERATURES)} temps × {len(eligible)} prompts)')
print(f'Durée estim.: ~{len(TEMPERATURES) * len(eligible) / RPM_LIMIT:.1f} min (séquentialisé par rate limit)')

## 3 — Fonction d'appel avec température paramétrable

In [ ]:
from probability_compat import decision_index, entropy_of, probabilities_of

from llm_module.adapters.base import ProviderClientError, ProviderServerError, ProviderParseError
from llm_module.worker.task_worker import _parse_ratelimit_reset_seconds

MAX_RETRIES    = 5
MAX_RETRY_WAIT = 300.0


def call_temperature(provider_name: str, temperature: float, entry: dict) -> list:
    """Appel brut — remplace le système prompt par PERSONA_SYSTEM_PROMPT, température forcée."""
    raw_messages = entry['messages']
    messages = [
        InternalMessage(role=raw_messages[0]['role'], content=PERSONA_SYSTEM_PROMPT),
        *[InternalMessage(role=m['role'], content=m['content']) for m in raw_messages[1:]],
    ]
    request = InternalRequest(
        provider=provider_name,
        messages=messages,
        response_schema=RESPONSE_SCHEMA,
        temperature=temperature,
    )
    llm_output, _, _ = get_adapter(provider_name).call(request)
    user_prompt = entry['messages'][1]['content']
    return [
        {
            'user_prompt':   user_prompt,
            'system_prompt': PERSONA_SYSTEM_PROMPT,
            'temperature':   temperature,
            'agent_id':      agent.agent_id,
            'chosen_index':  decision_index(agent),   # argmax de la distribution
                'entropy':       entropy_of(agent),       # hésitation déclarée
                'probabilities': probabilities_of(agent),
            'mode':          agent.mode,
            'reason':        agent.reason,
        }
        for agent in llm_output.agents
    ]


def call_temperature_with_retry(provider_name: str, temperature: float, entry: dict) -> list:
    parse_attempts = 0
    for attempt in range(MAX_RETRIES + 1):
        try:
            return call_temperature(provider_name, temperature, entry)
        except ProviderClientError as exc:
            if exc.status_code == 429 and attempt < MAX_RETRIES:
                wait = _parse_ratelimit_reset_seconds(exc.ratelimit_reset)
                if wait > MAX_RETRY_WAIT:
                    raise
                time.sleep(wait)
            else:
                raise
        except ProviderServerError:
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
            else:
                raise
        except ProviderParseError:
            if parse_attempts < 1:
                parse_attempts += 1
                time.sleep(2.0)
            else:
                raise

## 4 — Exécution parallèle (un thread par température, rate-limit global partagé)

In [ ]:
# ── Détection du mode reprise ──────────────────────────────────────────────
RESUME_MODE = OUTPUT_CSV.exists() and OUTPUT_CSV.stat().st_size > 100

if not RESUME_MODE:
    _results_dir = Path('results')
    if _results_dir.exists() and any(_results_dir.iterdir()):
        _ts      = datetime.now().strftime('%Y-%m-%d_%H%M%S')
        _archive = Path('results_archive') / _ts
        _archive.mkdir(parents=True, exist_ok=True)
        for _f in _results_dir.iterdir():
            shutil.copy2(_f, _archive / _f.name)
        print(f'Résultats précédents archivés → {_archive}')
    else:
        print('Aucun résultat précédent à archiver.')
else:
    print(f'Mode reprise — {OUTPUT_CSV} existe ({OUTPUT_CSV.stat().st_size // 1024} Ko)')

In [ ]:
# ── Rate-limiter global partagé entre tous les threads de température ──────
_RATE_LOCK    = threading.Lock()
_LAST_CALL_TS = [0.0]

def _rate_wait(rpm_limit: float) -> None:
    """Sérialise les appels pour ne pas dépasser rpm_limit toutes températures confondues."""
    min_interval = 60.0 / rpm_limit
    with _RATE_LOCK:
        now  = time.monotonic()
        wait = min_interval - (now - _LAST_CALL_TS[0])
        if wait > 0:
            time.sleep(wait)
        _LAST_CALL_TS[0] = time.monotonic()


_csv_lock      = threading.Lock()
_prompt_to_idx: dict = {}
_prompt_counter = 0

MAX_CONSECUTIVE_FAILURES = 10


def _get_prompt_idx(user_prompt: str) -> int:
    global _prompt_counter
    if user_prompt not in _prompt_to_idx:
        _prompt_to_idx[user_prompt] = _prompt_counter
        _prompt_counter += 1
    return _prompt_to_idx[user_prompt]


def process_temperature(temperature: float, entries: list, rpm_limit: int) -> list:
    rows = []
    consecutive_failures = 0
    temp_label = f'T={temperature:.1f}'
    bar = tqdm(entries, desc=temp_label, leave=True, position=TEMPERATURES.index(temperature))

    for entry in bar:
        user_prompt = entry['messages'][1]['content']

        if RESUME_MODE and (temperature, user_prompt) in _done_pairs:
            continue

        _rate_wait(rpm_limit)

        try:
            new_rows = call_temperature_with_retry(PROVIDER_NAME, temperature, entry)
            consecutive_failures = 0
            with _csv_lock:
                prompt_idx = _get_prompt_idx(new_rows[0]['user_prompt'])
                for r in new_rows:
                    r['prompt_idx'] = prompt_idx
                rows.extend(new_rows)
                write_header = not OUTPUT_CSV.exists()
                pd.DataFrame(new_rows).to_csv(OUTPUT_CSV, mode='a', header=write_header, index=False)
        except Exception as exc:
            consecutive_failures += 1
            bar.write(f'[{temp_label}] ECHEC ({consecutive_failures}/{MAX_CONSECUTIVE_FAILURES}): {exc}')
            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                bar.write(f'[{temp_label}] abandon après {MAX_CONSECUTIVE_FAILURES} échecs consécutifs')
                break

    return rows


# ── Initialisation ────────────────────────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

if RESUME_MODE and OUTPUT_CSV.exists():
    _existing_df = pd.read_csv(OUTPUT_CSV, engine='python')
    _done_pairs: set = set(
        zip(_existing_df['temperature'], _existing_df['user_prompt'])
    )
    for _up in _existing_df['user_prompt'].unique():
        _get_prompt_idx(_up)
    print(f'Reprise : {len(_done_pairs)} paires (température, prompt) déjà calculées')
else:
    _done_pairs = set()
    if OUTPUT_CSV.exists():
        OUTPUT_CSV.unlink()
    _prompt_to_idx.clear()
    _prompt_counter = 0

# ── Lancement (un thread par température) ─────────────────────────────────
all_rows: list = []

with ThreadPoolExecutor(max_workers=len(TEMPERATURES)) as executor:
    futures = {
        executor.submit(process_temperature, temp, eligible, RPM_LIMIT): temp
        for temp in TEMPERATURES
    }
    for future in as_completed(futures):
        temp = futures[future]
        try:
            all_rows.extend(future.result())
        except Exception as exc:
            print(f'[T={temp:.1f}] thread échoué: {exc}')

print(f'Terminé — {len(all_rows)} nouvelles lignes collectées')

## 5 — Aperçu des résultats

In [ ]:
df_preview = pd.read_csv(OUTPUT_CSV, engine='python')
print(f'Lignes totales      : {len(df_preview)}')
print(f'Températures        : {sorted(df_preview["temperature"].unique())}')
print(f'Prompts uniques     : {df_preview["user_prompt"].nunique()}')
print(f'Lignes par tempéra. : {df_preview.groupby("temperature").size().to_dict()}')
df_preview.head()

## 6 — Analyse des distributions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import entropy as sp_entropy

df_raw = pd.read_csv(OUTPUT_CSV, engine='python')

# Exclure les lignes en échec (chosen_index hors 1-5)
df_valid = df_raw[df_raw['chosen_index'].between(1, 5)].copy()
print(f'Lignes exclues (échecs) : {len(df_raw) - len(df_valid)}')

# Exclure les températures avec trop peu de réponses valides
MIN_RESPONSES = 10
counts = df_valid['temperature'].value_counts()
excluded_temps = counts[counts < MIN_RESPONSES].index.tolist()
if excluded_temps:
    print(f'Températures exclues (< {MIN_RESPONSES} réponses valides) : {excluded_temps}')
df = df_valid[~df_valid['temperature'].isin(excluded_temps)].copy()

# ── Palette officielle modes de transport ────────────────────────────────────
MODE_COLORS = {
    'Voiture': 'red',
    'Vélo':    'purple',
    'TC':      'green',
    'Marche':  'cyan',
    'Autre':   'gray',
}

def categorize_mode(m: str) -> str:
    if not isinstance(m, str):
        return 'Autre'
    ml = m.lower()
    if any(k in ml for k in ('voiture', 'car', 'conducteur')):
        return 'Voiture'
    if any(k in ml for k in ('vélo', 'velo', 'bicycle', 'cycling', 'vélib')):
        return 'Vélo'
    if any(k in ml for k in ('bus', 'metro', 'métro', 'tram', 'transit', 'transports en commun', 'public_transport')):
        return 'TC'
    if any(k in ml for k in ('marche', 'foot', 'walk')):
        return 'Marche'
    print(f'Mode non classifié (Autre) : {m!r}')
    return 'Autre'

df['mode_cat'] = df['mode'].apply(categorize_mode)
temps_order   = sorted(df['temperature'].unique())
temp_labels   = [f'T={t:.1f}' for t in temps_order]

print(f'\nDonnées analysées : {len(df)} lignes, {len(temps_order)} températures, {df["user_prompt"].nunique()} prompts uniques')
print('\nRéponses valides par température :')
print(df['temperature'].value_counts().sort_index().to_string())

### Graphique 1 — Distribution des modes de transport par température

In [ ]:
cats  = list(MODE_COLORS.keys())
pivot = (
    df.groupby(['temperature', 'mode_cat'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=cats, fill_value=0)
      .loc[temps_order]
)
pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(pct))
for cat in cats:
    vals = pct[cat].values
    ax.bar(range(len(pct)), vals, bottom=bottom,
           color=MODE_COLORS[cat], label=cat, width=0.65, edgecolor='white', linewidth=0.4)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 3:
            ax.text(i, b + v / 2, f'{v:.0f}%', ha='center', va='center',
                    fontsize=7.5, color='white', fontweight='bold')
    bottom += vals

ax.set_xticks(range(len(pct)))
ax.set_xticklabels(temp_labels, fontsize=10)
ax.set_xlabel('Température LLM')
ax.set_ylabel('Pourcentage (%)')
ax.set_ylim(0, 100)
ax.set_title('Distribution des modes de transport par température', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.85, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/g1_modes_par_temperature.png', dpi=150)
plt.show()

### Graphique 2 — Entropie de Shannon : diversité des choix en fonction de la température

In [ ]:
MAX_ENTROPY = np.log2(5)

def shannon_entropy(series):
    counts = series.value_counts(normalize=True)
    return sp_entropy(counts, base=2)

entropies = (
    df.groupby('temperature')['chosen_index']
      .apply(shannon_entropy)
      .reindex(temps_order)
)

fig, ax = plt.subplots(figsize=(9, 4))
colors_bar = ['#E53935' if e >= MAX_ENTROPY * 0.75 else '#EF9A9A' for e in entropies]
ax.bar(temp_labels, entropies, color=colors_bar, edgecolor='white', linewidth=0.5)
ax.axhline(MAX_ENTROPY, color='orange', linestyle='--', linewidth=1.2,
           label=f'Entropie max ({MAX_ENTROPY:.2f} bits)')
ax.set_xlabel('Température LLM')
ax.set_ylabel('Entropie de Shannon (bits)')
ax.set_title('Entropie des choix de mode en fonction de la température', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, MAX_ENTROPY * 1.15)

for i, v in enumerate(entropies):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontsize=8)

ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/g2_entropie_temperature.png', dpi=150)
plt.show()

### Graphique 3 — Accord pair-à-pair entre températures

In [ ]:
pivot_agree = (
    df.groupby(['user_prompt', 'temperature'])['chosen_index']
      .agg(lambda x: int(x.mode().iloc[0]))
      .unstack('temperature')
      .reindex(columns=temps_order)
)

n = len(temps_order)
agree_matrix = np.full((n, n), np.nan)

for i, ti in enumerate(temps_order):
    for j, tj in enumerate(temps_order):
        if i == j:
            agree_matrix[i, j] = 100.0
            continue
        col_i  = pivot_agree[ti].dropna()
        col_j  = pivot_agree[tj].dropna()
        common = col_i.index.intersection(col_j.index)
        if len(common) == 0:
            continue
        vi = col_i.loc[common].values.astype(int)
        vj = col_j.loc[common].values.astype(int)
        agree_matrix[i, j] = float((vi == vj).mean() * 100)

agree_df = pd.DataFrame(agree_matrix, index=temp_labels, columns=temp_labels)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    agree_df,
    annot=True, fmt='.0f', annot_kws={'size': 9},
    cmap='YlOrRd',
    vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Accord (%)'},
)
ax.set_title('Accord pair-à-pair entre températures\n(% de prompts avec le même choix d\'itinéraire)', fontsize=12, fontweight='bold')
plt.xticks(rotation=0, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('results/g3_accord_temperatures.png', dpi=150)
plt.show()

### Graphique 4 — Carte de chaleur des réponses par prompt × température

In [ ]:
pivot_resp = (
    df.groupby(['user_prompt', 'temperature'])['chosen_index']
      .agg(lambda x: x.mode()[0] if len(x) > 0 else np.nan)
      .unstack('temperature')
      .reindex(columns=temps_order)
)
pivot_resp = pivot_resp.loc[pivot_resp.median(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 12))
cmap   = plt.cm.get_cmap('RdYlGn', 5)
bounds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm   = mcolors.BoundaryNorm(bounds, cmap.N)

im = ax.imshow(pivot_resp.values, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')
ax.set_xticks(range(len(temps_order)))
ax.set_xticklabels(temp_labels, fontsize=10)
ax.set_yticks([])
ax.set_xlabel('Température LLM')
ax.set_ylabel(f'{len(pivot_resp)} prompts (triés par choix médian)', fontsize=10)
ax.set_title('Choix par prompt et par température (couleur = itinéraire 1–5)', fontsize=12, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, ticks=[1, 2, 3, 4, 5], pad=0.02)
cbar.ax.set_yticklabels(['1', '2', '3', '4', '5'])
cbar.set_label('Itinéraire choisi')

plt.tight_layout()
plt.savefig('results/g4_heatmap_prompts_temperature.png', dpi=150)
plt.show()

### Graphique 5 — Sensibilité des prompts à la température

Pour chaque prompt, on mesure combien de choix distincts ont été produits à travers toutes les températures testées.  
Un prompt **invariant** (1 seul choix) est insensible à la température.  
Un prompt **très sensible** (≥4 choix) voit son résultat fortement influencé.

In [ ]:
pivot_var = (
    df.groupby(['user_prompt', 'temperature'])['chosen_index']
      .agg(lambda x: x.mode()[0] if len(x) > 0 else np.nan)
      .unstack('temperature')
      .reindex(columns=temps_order)
)

n_distinct    = pivot_var.nunique(axis=1).dropna()
std_per_prompt = pivot_var.std(axis=1).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gauche : distribution du nombre de choix distincts
ax = axes[0]
counts_distinct = n_distinct.value_counts().sort_index()
ax.bar(counts_distinct.index.astype(int), counts_distinct.values,
       color='#5C6BC0', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Nombre de choix distincts à travers les températures')
ax.set_ylabel('Nombre de prompts')
ax.set_title('Sensibilité des prompts à la température\n(nombre de choix distincts)', fontsize=11, fontweight='bold')
ax.set_xticks(range(1, int(n_distinct.max()) + 1))
ax.spines[['top', 'right']].set_visible(False)
for x, y in zip(counts_distinct.index.astype(int), counts_distinct.values):
    ax.text(x, y + 0.3, str(y), ha='center', va='bottom', fontsize=9)

# Droite : histogramme de l'écart-type par prompt
ax = axes[1]
ax.hist(std_per_prompt.dropna(), bins=20, color='#26A69A', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Écart-type du chosen_index (par prompt, sur toutes températures)')
ax.set_ylabel('Nombre de prompts')
ax.set_title('Variabilité intra-prompt due à la température\n(std du chosen_index)', fontsize=11, fontweight='bold')
ax.axvline(std_per_prompt.mean(), color='red', linestyle='--', linewidth=1.2,
           label=f'Moyenne = {std_per_prompt.mean():.2f}')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/g5_sensibilite_prompts.png', dpi=150)
plt.show()

print(f'Prompts invariants (1 seul choix distinct)  : {(n_distinct == 1).sum()} / {len(n_distinct)}')
print(f'Prompts très sensibles (>= 4 choix distincts): {(n_distinct >= 4).sum()} / {len(n_distinct)}')
print(f'Std moyen par prompt : {std_per_prompt.mean():.2f}')

### Graphique 6 — Verbosité des justifications par température

In [ ]:
verbosity     = df.groupby('temperature')['reason'].apply(lambda s: s.str.len().mean()).reindex(temps_order)
verbosity_std = df.groupby('temperature')['reason'].apply(lambda s: s.str.len().std()).reindex(temps_order)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(temps_order))
ax.plot(x, verbosity.values, marker='o', linewidth=2, color='#7B1FA2', zorder=3)
ax.fill_between(x, verbosity.values, alpha=0.12, color='#7B1FA2')
ax.errorbar(x, verbosity.values, yerr=verbosity_std.values,
            fmt='none', color='#7B1FA2', capsize=4, alpha=0.4)

ax.set_xticks(x)
ax.set_xticklabels(temp_labels, fontsize=10)
ax.set_xlabel('Température LLM')
ax.set_ylabel('Longueur moyenne de la justification (caractères)')
ax.set_title('Verbosité des justifications en fonction de la température', fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/g6_verbosite_temperature.png', dpi=150)
plt.show()

print('\nVerbosité par température :')
summary_v = pd.DataFrame({'longueur_moy': verbosity, 'longueur_std': verbosity_std})
print(summary_v.to_string())